# DAIMS — Check and document a dataset before machine learning

Start with a fictional clinic export and its data dictionary. Run the DAIMS checklist, trace three issues to the source notes, then review the corrected files and save the checklist.

**DAIMS** stands for Datasheets for AI and Medical Datasets. Its automated checks are part of a 24-point checklist that also includes manual review. This notebook uses the same checks as the Streamlit app, with the notebook providing the files and the app producing the checklist.

[DAIMS on GitHub](https://github.com/PERSIMUNE/DAIMS) · [Open the DAIMS app](https://daims-app.streamlit.app/) · [Methods paper](https://arxiv.org/abs/2501.14094)

Enable Internet in Kaggle to download the pinned DAIMS source and example files.

In [ ]:
from contextlib import ExitStack
from pathlib import Path
from unittest.mock import MagicMock, patch
from urllib.request import urlretrieve
import runpy
import sys
from importlib.util import find_spec
import subprocess

missing = [name for name in ("xlsxwriter", "openpyxl") if find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

import pandas as pd
from IPython.display import Markdown, display

source = "https://raw.githubusercontent.com/PERSIMUNE/DAIMS/699bab9d53ba88c9f9794258eef2ca3922e8f0a4"
sample = "examples/synthetic-clinic"
files = Path("daims-example")
files.mkdir(exist_ok=True)

urlretrieve(f"{source}/app.py", files / "app.py")
for filename in (
    "starter-data.csv", "starter-dictionary.csv", "source-records.csv",
    "reviewed-data.csv", "reviewed-dictionary.csv", "datasheet.md",
):
    urlretrieve(f"{source}/{sample}/{filename}", files / filename)

## 1. Inspect the starter files

Each row represents a fictional visit. The data dictionary describes each variable's role, type, units, and permitted values. Here, `follow_up` is the outcome.

In [ ]:
display(pd.read_csv(files / "starter-data.csv", keep_default_na=False))
display(pd.read_csv(files / "starter-dictionary.csv", keep_default_na=False))

## 2. Run the DAIMS checks

The adapter below supplies the two files to the original DAIMS app and saves the Excel checklist it generates. Manual items stay **Unknown** until a reviewer answers them.

In [ ]:
def run_checklist(stage):
    interface = MagicMock()
    interface.columns.side_effect = lambda layout: [
        MagicMock() for _ in range(layout if isinstance(layout, int) else len(layout))
    ]
    interface.radio.return_value = "Unknown"
    interface.text_area.return_value = ""
    interface.button.return_value = False

    with ExitStack() as stack:
        interface.file_uploader.side_effect = [
            stack.enter_context((files / f"{stage}-{kind}.csv").open("rb"))
            for kind in ("data", "dictionary")
        ]
        stack.enter_context(patch.dict(sys.modules, {"streamlit": interface}))
        runpy.run_path(str(files / "app.py"))

    workbook = interface.download_button.call_args.kwargs["data"]
    output = files / f"{stage}-checklist.xlsx"
    output.write_bytes(workbook.getvalue())
    return pd.read_excel(output, index_col="#")


starter = run_checklist("starter")
display(starter.loc[starter["Mark if true"].eq("Fail"), ["Item", "Notification"]])

## 3. Review the source notes

- **Check 8 · Missing entries.** The starter export omits the blood pressure at visit `v08`. The fictional source note records 132 mmHg.
- **Check 10 · Data dictionary.** The dictionary leaves the unit for `systolic_bp` blank. The source note specifies mmHg.
- **Check 12 · Categories.** The starter export uses `rtne` for visit `v04`. The source note gives `routine`, a category already listed in the dictionary.

The reviewed files apply those three corrections. With research data, record the evidence for corrections and document how you handle any unresolved entries.

In [ ]:
display(pd.read_csv(files / "source-records.csv"))
reviewed = run_checklist("reviewed")
comparison = pd.concat(
    [starter["Mark if true"].rename("Starter"), reviewed["Mark if true"].rename("Reviewed")],
    axis=1,
)
display(comparison.loc[[8, 10, 12]])
display(reviewed[["Mark if true", "Automatically checked", "Item"]])

## 4. Save the checklist and datasheet

The notebook writes both Excel checklists to `daims-example/`. The example datasheet records the dataset's purpose, its fictional source, and the changes made during review. Use the full DAIMS template to document your own project.

[Full DAIMS datasheet template](https://github.com/PERSIMUNE/DAIMS/blob/main/DAIMS_DatasetName_DDMMYYYY.md)

In [ ]:
display(Markdown((files / "datasheet.md").read_text().split("For a real dataset,")[0]))

## Next step · Model with MAIT

The MAIT quick demo uses a separate public dataset to show model evaluation and an interactive workflow for interpreting feature contributions with SHAP.

[Run the MAIT demo on Kaggle](https://www.kaggle.com/code/remyz5/mait-explainable-ai-with-mixed-clinical-data)